In [1]:
# NOTE: Due to the stochastic nature of Genetic Algorithms,
# exact numerical reproduction may vary slightly across hardware
# and software environments. The results reported in the paper
# were obtained on Kaggle with GPU P100.
# Seed is fixed at 42 for best reproducibility.


import random
import numpy as np
import torch
import os

def set_seed(seed=42):
    # Python random
    random.seed(seed)

    # Numpy
    np.random.seed(seed)

    # PyTorch CPU
    torch.manual_seed(seed)

    # PyTorch GPU
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Ensures deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # For hash-based ops
    os.environ["PYTHONHASHSEED"] = str(seed)

# Set seed
set_seed(42)

In [2]:
!pip install transformers datasets faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.4 MB/s eta 0:00:00:00:0100:01


In [3]:
import pandas as pd
import faiss

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# HuggingFace
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

# PyTorch utilities
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn

# Device setup (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer once (used for all datasets)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
# Tokenization Function
# Converts raw text to input_ids and attention_mask

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

# ISOT

In [5]:
# Load ISOT Dataset

ISOT_PATH = "/kaggle/input/datasets/isharma28/isot-data/"

fake_df = pd.read_csv(ISOT_PATH + "Fake.csv")
real_df = pd.read_csv(ISOT_PATH + "True.csv")

fake_df["label"] = 0
real_df["label"] = 1

isot_df = pd.concat([fake_df, real_df], ignore_index=True)
isot_df = isot_df[["text", "label"]].dropna().drop_duplicates()


# First split → Train (80%) and Temp (20%)
isot_train_df, isot_temp_df = train_test_split(
    isot_df,
    test_size=0.2,
    stratify=isot_df["label"],
    random_state=42
)

# Split Temp → Validation (10%) and Test (10%)
isot_validation_df, isot_test_df = train_test_split(
    isot_temp_df,
    test_size=0.5,
    stratify=isot_temp_df["label"],
    random_state=42
)

print("\nSplit sizes:")
print("Train:", len(isot_train_df))
print("Validation:", len(isot_validation_df))
print("Test:", len(isot_test_df))


# Convert to HuggingFace Dataset

isot_train_dataset = Dataset.from_pandas(isot_train_df)
isot_validation_dataset = Dataset.from_pandas(isot_validation_df)
isot_test_dataset = Dataset.from_pandas(isot_test_df)


# Tokenize
isot_train_dataset = isot_train_dataset.map(tokenize_batch, batched=True)
isot_validation_dataset = isot_validation_dataset.map(tokenize_batch, batched=True)
isot_test_dataset = isot_test_dataset.map(tokenize_batch, batched=True)

# Set format
isot_train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

isot_validation_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

isot_test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

# Initialize RoBERTa

isot_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
).to(device)


# Training Arguments

isot_args = TrainingArguments(
    output_dir="./isot_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)


# Train

isot_trainer = Trainer(
    model=isot_model,
    args=isot_args,
    train_dataset=isot_train_dataset,
    eval_dataset=isot_validation_dataset
)

isot_trainer.train()

# Evaluate on Validation set

print("\nValidation Results:")

validation_results = isot_trainer.evaluate(
    eval_dataset=isot_validation_dataset
)

print(validation_results)



Split sizes:
Train: 30917
Validation: 3865
Test: 3865


Map:   0%|          | 0/30917 [00:00<?, ? examples/s]

Map:   0%|          | 0/3865 [00:00<?, ? examples/s]

Map:   0%|          | 0/3865 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation Results:


{'eval_loss': 0.0030557657591998577, 'eval_runtime': 26.8895, 'eval_samples_per_second': 143.736, 'eval_steps_per_second': 9.0, 'epoch': 3.0}


In [7]:
# 11. Evaluate on Test set (final unbiased evaluation)

print("\nTest Results:")

test_predictions = isot_trainer.predict(
    isot_test_dataset
)

test_preds = np.argmax(test_predictions.predictions, axis=1)

test_accuracy = accuracy_score(
    test_predictions.label_ids,
    test_preds
)

print("\nISOT Test Accuracy:", test_accuracy)




Test Results:



ISOT Test Accuracy: 1.0


In [8]:
# Save final model checkpoint

isot_trainer.save_model("./isot_results")

print("\nISOT model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


ISOT model saved successfully.


In [9]:
import os
print(os.listdir("./isot_results"))


['checkpoint-3866', 'training_args.bin', 'model.safetensors', 'checkpoint-5799', 'config.json', 'checkpoint-1933']


# COVID cross-validation

In [10]:
# Load COVID dataset
covid_train = pd.read_csv("/kaggle/input/datasets/isharma28/covid-data/Copy of Constraint_English_Train - Sheet1.csv")
covid_val = pd.read_csv("/kaggle/input/datasets/isharma28/covid-data/Constraint_English_Val - Sheet1.csv")

covid_train["label"] = covid_train["label"].map({"fake":0,"real":1})
covid_val["label"] = covid_val["label"].map({"fake":0,"real":1})

# Remove duplicates
covid_train = covid_train.drop_duplicates()
covid_val = covid_val.drop_duplicates()

# Reset index
covid_train = covid_train.reset_index(drop=True)
covid_val = covid_val.reset_index(drop=True)

# Convert pandas DataFrame → HuggingFace Dataset with correct column name "labels"

from datasets import Dataset

covid_train_dataset = Dataset.from_dict({

    "text": covid_train["tweet"].values,
    "labels": covid_train["label"].values   

})

covid_val_dataset = Dataset.from_dict({

    "text": covid_val["tweet"].values,
    "labels": covid_val["label"].values   

})


# Tokenize COVID dataset using RoBERTa tokenizer

covid_train_dataset = covid_train_dataset.map(
    tokenize_batch,
    batched=True
)

covid_val_dataset = covid_val_dataset.map(
    tokenize_batch,
    batched=True
)

# Convert dataset into PyTorch tensor format for Trainer and GA optimization

covid_train_dataset.set_format(

    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

covid_val_dataset.set_format(

    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/6420 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

In [11]:
# Install and import required libraries for Genetic Algorithm based layer optimization

!pip install pygad

import numpy as np
import torch
import pygad

from transformers import (
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import f1_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 3.3 MB/s eta 0:00:00


In [12]:
# Freeze/unfreeze RoBERTa layers based on GA chromosome for COVID domain adaptation

def apply_covid_layer_mask_to_isot_roberta(model, covid_layer_mask):

    # Always freeze embedding layer (general language features)
    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False

    # Apply GA mask to encoder layers
    for layer_index, layer in enumerate(model.roberta.encoder.layer):

        if covid_layer_mask[layer_index] == 1:

            # Unfreeze this layer
            for param in layer.parameters():
                param.requires_grad = True

        else:

            # Freeze this layer
            for param in layer.parameters():
                param.requires_grad = False

    # Always unfreeze classifier head (required for COVID adaptation)
    for param in model.classifier.parameters():
        param.requires_grad = True

In [13]:
# Train ISOT-initialized RoBERTa with selected layer mask on COVID dataset
# and return validation F1 score as GA fitness value

def covid_layer_unfreeze_fitness_function(
    ga_instance,
    covid_layer_mask_solution,
    solution_index
):

    covid_layer_mask_solution = covid_layer_mask_solution.astype(int)

    # Load ISOT-initialized RoBERTa checkpoint
    covid_ga_model = RobertaForSequenceClassification.from_pretrained(
        "./isot_results/checkpoint-5799",
        num_labels=2
    ).to(device)

    # Apply GA layer mask
    apply_covid_layer_mask_to_isot_roberta(
        covid_ga_model,
        covid_layer_mask_solution
    )

    # Fast training config for GA fitness evaluation
    covid_ga_training_args = TrainingArguments(

        output_dir="./temp_covid_ga",
        num_train_epochs=1,   
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="no",
        report_to="none"
    )

    # Trainer instance
    covid_ga_trainer = Trainer(
        model=covid_ga_model,
        args=covid_ga_training_args,
        train_dataset=covid_train_dataset,
        eval_dataset=covid_val_dataset
    )

    # Train model briefly
    covid_ga_trainer.train()

    # Evaluate
    covid_ga_predictions = covid_ga_trainer.predict(
        covid_val_dataset
    )

    covid_ga_predicted_labels = np.argmax(
        covid_ga_predictions.predictions,
        axis=1
    )

    covid_ga_true_labels = covid_ga_predictions.label_ids

    covid_ga_f1_score = f1_score(
        covid_ga_true_labels,
        covid_ga_predicted_labels
    )

    print(
        "COVID Layer Mask:",
        covid_layer_mask_solution,
        "| Validation F1:",
        covid_ga_f1_score
    )

    return covid_ga_f1_score

In [14]:
# Configure GA parameters to optimize RoBERTa layer unfreezing for COVID dataset

covid_layer_ga_instance = pygad.GA(
    num_generations=2,  #run optimization for 1 generation
    sol_per_pop=5,  # solution per generation
    num_parents_mating=2, # choose best 2 mask, later used for creating news mask by mutation for next generation
    fitness_func=covid_layer_unfreeze_fitness_function,# Loads ISOT initialized RoBERTa, Applies mask, Trains briefly on COVID dataset,Computes F1 score ,Returns F1 score to GA
    num_genes=12,   # 12 RoBERTa encoder layers
    gene_type=int,
    gene_space=[0, 1],     # each layer freeze/un-freeze 
    mutation_percent_genes=20  #20% of genes mutate during mutation
    
)

In [15]:
# Track and display Genetic Algorithm progress for COVID layer optimization

# define global variable
covid_ga_execution_counter = 0

covid_ga_total_executions = (
    covid_layer_ga_instance.num_generations *
    covid_layer_ga_instance.sol_per_pop
)

covid_best_layer_mask_global = None
covid_best_f1_global = -1


# Track best solution  (NO retraining later)

def covid_ga_on_generation_callback(ga_instance):

    global covid_best_layer_mask_global
    global covid_best_f1_global

    best_idx = np.argmax(ga_instance.last_generation_fitness)

    best_mask = ga_instance.population[best_idx]
    best_f1 = ga_instance.last_generation_fitness[best_idx]

    if best_f1 > covid_best_f1_global:

        covid_best_f1_global = best_f1
        covid_best_layer_mask_global = best_mask.copy()

    total = ga_instance.num_generations * ga_instance.sol_per_pop
    completed = ga_instance.generations_completed * ga_instance.sol_per_pop
    remaining = total - completed

    print("\n========== COVID GA PROGRESS ==========")
    print(f"Generation: {ga_instance.generations_completed}/{ga_instance.num_generations}")
    print(f"Completed: {completed}/{total}")
    print(f"Remaining: {remaining}")
    print(f"Best F1 so far: {covid_best_f1_global:.4f}")
    print("=======================================\n")


In [16]:
# Connect progress tracking callback to COVID GA instance

covid_layer_ga_instance.on_generation = covid_ga_on_generation_callback

In [19]:
# Execute Genetic Algorithm to find optimal RoBERTa layers for COVID adaptation

covid_layer_ga_instance.run()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.225995


COVID Layer Mask: [1 0 1 0 1 1 0 0 1 0 1 0] | Validation F1: 0.9399048854301773


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.206982


...

Epoch,Training Loss,Validation Loss
1,No log,0.197297


COVID Layer Mask: [1 1 1 0 1 1 1 0 1 1 0 1] | Validation F1: 0.9517181383210092


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.231309


COVID Layer Mask: [0 1 1 0 1 1 1 0 0 1 1 1] | Validation F1: 0.9272030651340997

========== COVID GA PROGRESS ==========
Generation: 4/2
Completed: 20/10
Remaining: -10
Best F1 so far: 0.9565



In [20]:
# Retrieve best layer mask safely (instant, no retraining)

covid_best_layer_mask = covid_best_layer_mask_global.astype(int)
covid_best_f1 = covid_best_f1_global

print("\nFINAL BEST RESULT")
print("Best COVID Layer Mask:", covid_best_layer_mask)
print("Best COVID Validation F1:", covid_best_f1)


FINAL BEST RESULT
Best COVID Layer Mask: [1 1 1 1 1 1 1 0 0 1 1 1]
Best COVID Validation F1: 0.9564835164835165


In [21]:

# Compute accuracy, F1, precision, recall for COVID fake news classification

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    covid_logits, covid_true_labels = eval_pred
    
    covid_predicted_labels = covid_logits.argmax(axis=1)

    covid_precision, covid_recall, covid_f1, _ = precision_recall_fscore_support(
        covid_true_labels,
        covid_predicted_labels,
        average="binary"
    )

    covid_accuracy = accuracy_score(
        covid_true_labels,
        covid_predicted_labels
    )

    return {
        "accuracy": covid_accuracy,
        "f1": covid_f1,
        "precision": covid_precision,
        "recall": covid_recall
    }

In [22]:
# Train final ISOT-initialized RoBERTa on COVID dataset using optimal GA layer configuration

covid_final_ga_model = RobertaForSequenceClassification.from_pretrained(
    "./isot_results/checkpoint-5799",
    num_labels=2
).to(device)

apply_covid_layer_mask_to_isot_roberta(
    covid_final_ga_model,
    covid_best_layer_mask
)

covid_final_training_args = TrainingArguments(

    output_dir="./covid_ga_optimized_model",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

covid_final_trainer = Trainer(

    model=covid_final_ga_model,
    args=covid_final_training_args,
    train_dataset=covid_train_dataset,
    eval_dataset=covid_val_dataset,
    compute_metrics=compute_metrics
)

covid_final_trainer.train()
covid_final_results = covid_final_trainer.evaluate()

print("Final COVID GA Optimized Results:", covid_final_results)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.212415,0.947664,0.952096,0.913793,0.993750
2,0.377110,0.164222,0.967290,0.969352,0.951031,0.988393
3,0.104921,0.143665,0.972897,0.974517,0.959343,0.990179
4,0.054910,0.144805,0.973832,0.975309,0.963415,0.987500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Final COVID GA Optimized Results: {'eval_loss': 0.1436714231967926, 'eval_accuracy': 0.9728971962616823, 'eval_f1': 0.9745166959578208, 'eval_precision': 0.9593425605536332, 'eval_recall': 0.9901785714285715, 'eval_runtime': 14.9431, 'eval_samples_per_second': 143.21, 'eval_steps_per_second': 8.967, 'epoch': 4.0}


In [23]:
#save optimized model
covid_final_trainer.save_model("./covid_ga_optimized_roberta")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
#save best layer mask
np.save(
    "covid_ga_best_layer_mask.npy",
    covid_best_layer_mask
)

In [25]:
#COVID test dataset for final unbiased evaluation

covid_test = pd.read_csv("/kaggle/input/datasets/isharma28/covid-val-dataset/english_test_with_labels - Sheet1.csv")

# Convert labels to numeric
covid_test["label"] = covid_test["label"].map({
    "fake": 0,
    "real": 1
})


# Convert COVID validation pandas DataFrame → HuggingFace Dataset

from datasets import Dataset

covid_test_dataset = Dataset.from_dict({

    "text": covid_test["tweet"].values,

    "labels": covid_test["label"].values

})


# Tokenize COVID validation dataset using RoBERTa tokenizer

covid_test_dataset = covid_test_dataset.map(
    tokenize_batch,
    batched=True
)

# Convert COVID validation dataset into PyTorch tensor format

covid_test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

In [26]:
# Evaluate final GA-optimized model on unseen COVID validation dataset

covid_test_results = covid_final_trainer.evaluate(
    eval_dataset=covid_test_dataset
)

print("\nFINAL COVID TEST RESULTS:")
print("COVID Validation Accuracy:", covid_test_results["eval_accuracy"])
print("COVID Validation F1:", covid_test_results["eval_f1"])
print("COVID Validation Precision:", covid_test_results["eval_precision"])
print("COVID Validation Recall:", covid_test_results["eval_recall"])


FINAL COVID TEST RESULTS:
COVID Validation Accuracy: 0.9728971962616823
COVID Validation F1: 0.9744493392070485
COVID Validation Precision: 0.9617391304347827
COVID Validation Recall: 0.9875


In [27]:

# Generate detailed classification report on COVID validation dataset

from sklearn.metrics import classification_report

covid_test_predictions = covid_final_trainer.predict(
    covid_test_dataset
)

covid_test_predicted_labels = covid_test_predictions.predictions.argmax(axis=1)

covid_test_true_labels = covid_test_predictions.label_ids

print("\nCOVID Test Classification Report:\n")

print(
    classification_report(
        covid_test_true_labels,
        covid_test_predicted_labels,
        target_names=["Fake", "Real"]
    )
)


COVID Test Classification Report:

              precision    recall  f1-score   support

        Fake       0.99      0.96      0.97      1020
        Real       0.96      0.99      0.97      1120

    accuracy                           0.97      2140
   macro avg       0.97      0.97      0.97      2140
weighted avg       0.97      0.97      0.97      2140



# Gossipcop

In [28]:
# Load GossipCop dataset from single CSV file

gossipcop = pd.read_csv("/kaggle/input/datasets/isharma28/gossipcop-data/gossipcop.csv")

print(gossipcop.columns)
print(gossipcop.head())

Index(['Unnamed: 0', 'text', 'label'], dtype='object')
   Unnamed: 0                                               text label
0           0  Star magazine has released an explosive report...  fake
1           1  Earlier this year, the buzz around Megyn Kelly...  fake
2           2  For the first time since his involvement in a ...  fake
3           3  Those heels were cute, but they didn't last lo...  fake
4           4  American reality television personality and re...  fake


In [29]:
# Convert GossipCop labels from fake/real → 0/1

gossipcop["label"] = gossipcop["label"].map({
    "fake": 0,
    "real": 1
})
# Drop rows where text is missing
gossipcop = gossipcop.dropna(subset=["text"])

# Drop rows where label is missing 
gossipcop = gossipcop.dropna(subset=["label"])

# Remove duplicate rows
gossipcop = gossipcop.drop_duplicates()

# Reset index
gossipcop = gossipcop.reset_index(drop=True)
# Verify
print(gossipcop["label"].value_counts())

label
1    15223
0     4784
Name: count, dtype: int64


In [30]:
# First split: Train (70%) and Temp (30%)

from sklearn.model_selection import train_test_split

gossipcop_train, gossipcop_temp = train_test_split(

    gossipcop,
    test_size=0.3,
    stratify=gossipcop["label"],
    random_state=42
)


# Second split: Test (15%) and Validation (15%)

gossipcop_test, gossipcop_validation = train_test_split(

    gossipcop_temp,
    test_size=0.5,
    stratify=gossipcop_temp["label"],
    random_state=42
)

print("Train size:", len(gossipcop_train))
print("Test size:", len(gossipcop_test))
print("Validation size:", len(gossipcop_validation))

Train size: 14004
Test size: 3001
Validation size: 3002


In [31]:

# Convert GossipCop splits → HuggingFace Dataset

from datasets import Dataset

gossipcop_train_dataset = Dataset.from_dict({

    "text": gossipcop_train["text"].values,
    "labels": gossipcop_train["label"].values
})

gossipcop_test_dataset = Dataset.from_dict({

    "text": gossipcop_test["text"].values,
    "labels": gossipcop_test["label"].values
})

gossipcop_validation_dataset = Dataset.from_dict({

    "text": gossipcop_validation["text"].values,
    "labels": gossipcop_validation["label"].values
})

In [32]:
# Tokenize GossipCop datasets

gossipcop_train_dataset = gossipcop_train_dataset.map(
    tokenize_batch,
    batched=True
)

gossipcop_test_dataset = gossipcop_test_dataset.map(
    tokenize_batch,
    batched=True
)

gossipcop_validation_dataset = gossipcop_validation_dataset.map(
    tokenize_batch,
    batched=True
)

Map:   0%|          | 0/14004 [00:00<?, ? examples/s]

Map:   0%|          | 0/3001 [00:00<?, ? examples/s]

Map:   0%|          | 0/3002 [00:00<?, ? examples/s]

In [33]:
# Convert to PyTorch format

gossipcop_train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

gossipcop_test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

gossipcop_validation_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [34]:
# Apply GA layer mask to ISOT-initialized RoBERTa for GossipCop

def apply_gossipcop_layer_mask_to_isot_roberta(model, gossipcop_layer_mask):

    # Freeze embeddings
    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False

    # Apply mask to encoder layers
    for layer_index, layer in enumerate(model.roberta.encoder.layer):

        if gossipcop_layer_mask[layer_index] == 1:

            for param in layer.parameters():
                param.requires_grad = True

        else:

            for param in layer.parameters():
                param.requires_grad = False

    # Always train classifier
    for param in model.classifier.parameters():
        param.requires_grad = True

In [35]:
# GA fitness function for GossipCop layer optimization

def gossipcop_layer_unfreeze_fitness_function(
    ga_instance,
    gossipcop_layer_mask_solution,
    solution_index
):

    gossipcop_layer_mask_solution = gossipcop_layer_mask_solution.astype(int)

    gossipcop_ga_model = RobertaForSequenceClassification.from_pretrained(
        "./isot_results/checkpoint-5799",
        num_labels=2
    ).to(device)

    apply_gossipcop_layer_mask_to_isot_roberta(
        gossipcop_ga_model,
        gossipcop_layer_mask_solution
    )

    gossipcop_ga_training_args = TrainingArguments(

        output_dir="./temp_gossipcop_ga",
        num_train_epochs=1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="no",
        report_to="none"
    )

    gossipcop_ga_trainer = Trainer(

        model=gossipcop_ga_model,
        args=gossipcop_ga_training_args,
        train_dataset=gossipcop_train_dataset,
        eval_dataset=gossipcop_test_dataset
    )

    gossipcop_ga_trainer.train()

    gossipcop_predictions = gossipcop_ga_trainer.predict(
        gossipcop_test_dataset
    )

    gossipcop_predicted_labels = np.argmax(
        gossipcop_predictions.predictions,
        axis=1
    )

    gossipcop_true_labels = gossipcop_predictions.label_ids

    gossipcop_f1_score = f1_score(
        gossipcop_true_labels,
        gossipcop_predicted_labels
    )

    print(
        "GossipCop Layer Mask:",
        gossipcop_layer_mask_solution,
        "| Validation F1:",
        gossipcop_f1_score
    )

    return gossipcop_f1_score

In [36]:
# Initialize GA optimizer for GossipCop

import pygad

gossipcop_layer_ga_instance = pygad.GA(
    num_generations=2,
    sol_per_pop=5,
    num_parents_mating=3,
    fitness_func=gossipcop_layer_unfreeze_fitness_function,
    num_genes=12,
    gene_type=int,
    gene_space=[0, 1],
    mutation_percent_genes=20
)

In [37]:
# Track and display Genetic Algorithm progress for GossipCop layer optimization

# Counter to track executions
gossipcop_ga_execution_counter = 0

# Total executions
gossipcop_ga_total_executions = (
    gossipcop_layer_ga_instance.num_generations *
    gossipcop_layer_ga_instance.sol_per_pop
)

gossipcop_best_layer_mask_global = None
gossipcop_best_f1_global = -1


#  GA progress callback for GossipCop optimization 
def gossipcop_ga_on_generation_callback(ga_instance):

    global gossipcop_best_layer_mask_global
    global gossipcop_best_f1_global

    # Find best solution in current generation
    best_idx = np.argmax(ga_instance.last_generation_fitness)

    best_mask = ga_instance.population[best_idx]
    best_f1 = ga_instance.last_generation_fitness[best_idx]

    # Update global best if improved
    if best_f1 > gossipcop_best_f1_global:

        gossipcop_best_f1_global = best_f1
        gossipcop_best_layer_mask_global = best_mask.copy()

    # Automatic progress calculation
    total = ga_instance.num_generations * ga_instance.sol_per_pop
    completed = ga_instance.generations_completed * ga_instance.sol_per_pop
    remaining = total - completed

    print("\n========== GossipCop GA PROGRESS ==========")

    print(f"Generation: {ga_instance.generations_completed}/{ga_instance.num_generations}")

    print(f"Completed: {completed}/{total}")

    print(f"Remaining: {remaining}")

    print(f"Best GossipCop Validation F1 so far: {gossipcop_best_f1_global:.4f}")

    print("==========================================\n")

In [38]:
#progress tracking callback to GossipCop GA instance

gossipcop_layer_ga_instance.on_generation = gossipcop_ga_on_generation_callback

In [39]:
# Run GossipCop GA optimization 
gossipcop_layer_ga_instance.run()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.446498


GossipCop Layer Mask: [0 0 1 0 0 0 0 0 1 0 0 0] | Validation F1: 0.8965233491051224


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.484926


...

Epoch,Training Loss,Validation Loss
1,No log,0.441987


GossipCop Layer Mask: [0 0 1 1 0 0 0 0 0 0 1 0] | Validation F1: 0.8976282181228461


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.426738


GossipCop Layer Mask: [0 1 1 0 1 1 0 0 0 0 1 1] | Validation F1: 0.895802881603675

========== GossipCop GA PROGRESS ==========
Generation: 2/2
Completed: 10/10
Remaining: 0
Best GossipCop Validation F1 so far: 0.9073



In [40]:

# Retrieve best GossipCop mask safely without retraining

gossipcop_best_layer_mask = gossipcop_best_layer_mask_global.astype(int)

gossipcop_best_f1 = gossipcop_best_f1_global

print("Best GossipCop Layer Mask:", gossipcop_best_layer_mask)

print("Best GossipCop Validation F1:", gossipcop_best_f1)

Best GossipCop Layer Mask: [1 1 1 0 1 1 0 1 1 0 0 0]
Best GossipCop Validation F1: 0.9072765072765073


In [41]:
# Train final GA-optimized GossipCop model with live progress, loss, and F1 tracking

gossipcop_final_ga_model = RobertaForSequenceClassification.from_pretrained(
    "./isot_results/checkpoint-5799",
    num_labels=2
).to(device)

# Apply GA-discovered optimal layer mask
apply_gossipcop_layer_mask_to_isot_roberta(
    gossipcop_final_ga_model,
    gossipcop_best_layer_mask
)

# Training configuration with progress logging
gossipcop_final_training_args = TrainingArguments(

    output_dir="./gossipcop_ga_optimized_model",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    logging_strategy="steps",
    logging_steps=100,      # show progress every 100 steps
    disable_tqdm=False,     # enable progress bar
    report_to="none"
)

# Trainer with metrics tracking
gossipcop_final_trainer = Trainer(

    model=gossipcop_final_ga_model,
    args=gossipcop_final_training_args,
    train_dataset=gossipcop_train_dataset,
    eval_dataset=gossipcop_test_dataset,
    compute_metrics=compute_metrics
)

# Start training 
gossipcop_final_trainer.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.361494,0.409084,0.860380,0.911097,0.883903,0.940018
2,0.365655,0.392114,0.867044,0.917443,0.869753,0.970665
3,0.319799,0.352133,0.868377,0.917931,0.873468,0.967163
4,0.278500,0.361288,0.870377,0.917111,0.893317,0.942207


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3504, training_loss=0.3515937865188677, metrics={'train_runtime': 1243.7204, 'train_samples_per_second': 45.039, 'train_steps_per_second': 2.817, 'total_flos': 7369214438522880.0, 'train_loss': 0.3515937865188677, 'epoch': 4.0})

In [42]:
# Evaluate final model on unseen GossipCop validation dataset

gossipcop_final_results = gossipcop_final_trainer.evaluate(
    eval_dataset=gossipcop_validation_dataset
)

print("\nFinal GossipCop GA Optimized Results:")
print("Accuracy:", gossipcop_final_results["eval_accuracy"])
print("F1:", gossipcop_final_results["eval_f1"])
print("Precision:", gossipcop_final_results["eval_precision"])
print("Recall:", gossipcop_final_results["eval_recall"])


Final GossipCop GA Optimized Results:
Accuracy: 0.8667554963357762
F1: 0.9169090153718321
Precision: 0.8723320158102766
Recall: 0.9662872154115587


In [43]:
from sklearn.metrics import classification_report

preds = gossipcop_final_trainer.predict(gossipcop_validation_dataset)

y_pred = preds.predictions.argmax(axis=1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred, target_names=["Fake","Real"]))

              precision    recall  f1-score   support

        Fake       0.84      0.55      0.66       718
        Real       0.87      0.97      0.92      2284

    accuracy                           0.87      3002
   macro avg       0.85      0.76      0.79      3002
weighted avg       0.86      0.87      0.86      3002



# WelFake

In [44]:
# Load WELFake dataset

import pandas as pd

welfake = pd.read_csv(
    "/kaggle/input/datasets/isharma28/welfake/WELFake_Dataset.csv"
)

print("Dataset shape:", welfake.shape)
print(welfake.columns)
print(welfake.head())

Dataset shape: (72134, 4)
Index(['Unnamed: 0', 'title', 'text', 'label'], dtype='object')
   Unnamed: 0                                              title  \
0           0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1           1                                                NaN   
2           2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3           3  Bobby Jindal, raised Hindu, uses story of Chri...   
4           4  SATAN 2: Russia unvelis an image of its terrif...   

                                                text  label  
0  No comment is expected from Barack Obama Membe...      1  
1     Did they post their votes for Hillary already?      1  
2   Now, most of the demonstrators gathered last ...      1  
3  A dozen politically active pastors came here f...      0  
4  The RS-28 Sarmat missile, dubbed Satan 2, will...      1  


In [45]:
# Clean WELFake dataset, remove invalid rows

# Keep only text and label columns
welfake = welfake[["text", "label"]]

# Drop rows where text is missing
welfake = welfake.dropna(subset=["text"])

# Drop rows where label is missing
welfake = welfake.dropna(subset=["label"])

# Convert label to integer
welfake["label"] = welfake["label"].astype(int)

# Reset index (recommended after dropping rows)
welfake = welfake.reset_index(drop=True)

# Rename text column to content (for compatibility with your pipeline)
welfake = welfake.rename(columns={"text": "content"})

# Check label distribution
print("Label distribution:")
print(welfake["label"].value_counts())

# Check total rows
print("\nTotal rows after cleaning:", len(welfake))

Label distribution:
label
1    37067
0    35028
Name: count, dtype: int64

Total rows after cleaning: 72095


In [47]:
welfake_train, welfake_temp = train_test_split(

    welfake,
    test_size=0.3,
    stratify=welfake["label"],
    random_state=42
)

# Second split: test (15%) and validation (15%)
welfake_test, welfake_validation = train_test_split(

    welfake_temp,
    test_size=0.5,
    stratify=welfake_temp["label"],
    random_state=42
)

print("Train:", len(welfake_train))
print("Test:", len(welfake_test))
print("Validation:", len(welfake_validation))

Train: 50466
Test: 10814
Validation: 10815


In [48]:
#  Convert to HuggingFace dataset format

from datasets import Dataset

welfake_train_dataset = Dataset.from_dict({

    "text": welfake_train["content"].values,
    "labels": welfake_train["label"].values
})

welfake_test_dataset = Dataset.from_dict({

    "text": welfake_test["content"].values,
    "labels": welfake_test["label"].values
})

welfake_validation_dataset = Dataset.from_dict({

    "text": welfake_validation["content"].values,
    "labels": welfake_validation["label"].values
})

In [49]:
# Tokenize WELFake datasets

welfake_train_dataset = welfake_train_dataset.map(
    tokenize_batch,
    batched=True
)

welfake_test_dataset = welfake_test_dataset.map(
    tokenize_batch,
    batched=True
)

welfake_validation_dataset = welfake_validation_dataset.map(
    tokenize_batch,
    batched=True
)

Map:   0%|          | 0/50466 [00:00<?, ? examples/s]

Map:   0%|          | 0/10814 [00:00<?, ? examples/s]

Map:   0%|          | 0/10815 [00:00<?, ? examples/s]

In [50]:
# Convert to PyTorch format

welfake_train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

welfake_test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

welfake_validation_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)



In [51]:
# Apply GA layer mask to RoBERTa

def apply_welfake_layer_mask_to_isot_roberta(model, layer_mask):

    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False

    for i in range(12):

        if layer_mask[i] == 1:

            for param in model.roberta.encoder.layer[i].parameters():
                param.requires_grad = True

        else:

            for param in model.roberta.encoder.layer[i].parameters():
                param.requires_grad = False

    for param in model.classifier.parameters():
        param.requires_grad = True

In [52]:
# GA fitness function for WELFake

from transformers import RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score
import numpy as np

def welfake_layer_unfreeze_fitness_function(

    ga_instance,
    layer_mask_solution,
    solution_index
):

    layer_mask_solution = layer_mask_solution.astype(int)

    model = RobertaForSequenceClassification.from_pretrained(

        "./isot_results/checkpoint-5799",
        num_labels=2

    ).to(device)

    apply_welfake_layer_mask_to_isot_roberta(
        model,
        layer_mask_solution
    )

    args = TrainingArguments(

        output_dir="./temp_welfake_ga",
        num_train_epochs=1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="no",
        report_to="none"
    )

    trainer = Trainer(

        model=model,
        args=args,
        train_dataset=welfake_train_dataset,
        eval_dataset=welfake_test_dataset
    )

    trainer.train()

    preds = trainer.predict(welfake_test_dataset)

    pred_labels = np.argmax(preds.predictions, axis=1)

    f1 = f1_score(preds.label_ids, pred_labels)

    print("Mask:", layer_mask_solution, "| F1:", f1)

    return f1

In [53]:

# Store globally best WELFake layer mask safely (prevents retraining)

welfake_best_layer_mask_global = None
welfake_best_f1_global = -1

In [54]:

# Safe GA progress callback for WELFake optimization 

def welfake_ga_on_generation_callback(ga_instance):

    global welfake_best_layer_mask_global
    global welfake_best_f1_global

    # Find best solution in current generation
    best_idx = np.argmax(ga_instance.last_generation_fitness)

    best_mask = ga_instance.population[best_idx]
    best_f1 = ga_instance.last_generation_fitness[best_idx]

    # Update global best solution automatically
    if best_f1 > welfake_best_f1_global:

        welfake_best_f1_global = best_f1
        welfake_best_layer_mask_global = best_mask.copy()

    # Automatic progress calculation
    total = ga_instance.num_generations * ga_instance.sol_per_pop
    completed = ga_instance.generations_completed * ga_instance.sol_per_pop
    remaining = total - completed

    print("\n========== WELFake GA PROGRESS ==========")

    print(f"Generation: {ga_instance.generations_completed}/{ga_instance.num_generations}")

    print(f"Completed: {completed}/{total}")

    print(f"Remaining: {remaining}")

    print(f"Best WELFake Validation F1 so far: {welfake_best_f1_global:.4f}")

    print("=========================================\n")

In [55]:
# Run GA optimization

import pygad

welfake_layer_ga_instance = pygad.GA(

    num_generations=1,
    sol_per_pop=3,
    num_parents_mating=2,
    fitness_func=welfake_layer_unfreeze_fitness_function,
    on_generation=welfake_ga_on_generation_callback,
    num_genes=12,
    gene_type=int,
    gene_space=[0,1],
    mutation_percent_genes=20
    
)



In [56]:
# Execute WELFake GA optimization

welfake_layer_ga_instance.run()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.015994


Mask: [0 0 1 0 0 0 0 0 1 0 0 0] | F1: 0.9964849031095088


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.022034


...

Epoch,Training Loss,Validation Loss
1,No log,0.012484


Mask: [1 1 0 1 0 0 0 0 1 0 1 0] | F1: 0.9974797479747974


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.016582


Mask: [0 0 1 0 0 0 1 0 1 1 1 1] | F1: 0.9970294355927626

========== WELFake GA PROGRESS ==========
Generation: 1/1
Completed: 3/3
Remaining: 0
Best WELFake Validation F1 so far: 0.9978



In [57]:
# Retrieve best WELFake mask safely 

welfake_best_layer_mask = welfake_best_layer_mask_global.astype(int)

welfake_best_f1 = welfake_best_f1_global

print("Best WELFake Layer Mask:", welfake_best_layer_mask)

print("Best WELFake Validation F1:", welfake_best_f1)

Best WELFake Layer Mask: [1 0 0 1 0 0 1 0 1 1 1 0]
Best WELFake Validation F1: 0.9977511918683099


In [58]:
# Compute accuracy, precision, recall, and F1 score during evaluation

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )

    accuracy = accuracy_score(labels, preds)

    return {

        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [59]:
# Final training

welfake_final_model = RobertaForSequenceClassification.from_pretrained(

    "./isot_results/checkpoint-5799",
    num_labels=2

).to(device)

apply_welfake_layer_mask_to_isot_roberta(
    welfake_final_model,
    welfake_best_layer_mask
)

args = TrainingArguments(

    output_dir="./welfake_final",
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(

    model=welfake_final_model,
    args=args,
    train_dataset=welfake_train_dataset,
    eval_dataset=welfake_test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.047170,0.236571,0.938691,0.936610,0.999796,0.880935
2,0.000324,0.024990,0.996671,0.996757,0.998556,0.994964
3,0.000095,0.008617,0.998428,0.998470,0.999279,0.997662
4,0.012777,0.006774,0.998705,0.998740,0.999460,0.998022


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=25236, training_loss=0.026522301430985943, metrics={'train_runtime': 4620.7113, 'train_samples_per_second': 43.687, 'train_steps_per_second': 5.461, 'total_flos': 2.655632503959552e+16, 'train_loss': 0.026522301430985943, 'epoch': 4.0})

In [60]:
# Final evaluation on validation dataset

results = trainer.evaluate(
    eval_dataset=welfake_validation_dataset
)

print(results)

{'eval_loss': 0.00867975689470768, 'eval_accuracy': 0.9985205732778548, 'eval_f1': 0.9985601151907847, 'eval_precision': 0.9992795389048992, 'eval_recall': 0.9978417266187051, 'eval_runtime': 79.0683, 'eval_samples_per_second': 136.781, 'eval_steps_per_second': 17.099, 'epoch': 4.0}


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {

        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# Accuracy and F1 score

In [74]:


import pandas as pd

accuracy_table = pd.DataFrame({

    "Dataset": [

        "COVID",
        "GossipCop",
        "WELFake"
    ],

    

    "ISOT+GA Accuracy": [

        covid_final_results["eval_accuracy"],
        gossipcop_final_results["eval_accuracy"],
        results["eval_accuracy"]
    ],

    "ISOT+GA F1": [

        covid_final_results["eval_f1"],
        gossipcop_final_results["eval_f1"],
        results["eval_f1"]
    ]
})

accuracy_table

,Dataset,ISOT+GA Accuracy,ISOT+GA F1
0,COVID,0.972897,0.974517
1,GossipCop,0.866755,0.916909
2,WELFake,0.998521,0.998560
